# 02 — Features: definitions, Table 1, data checks

**Role A (features).** Feature logic lives in `src/features.py`; this notebook documents the three nested feature sets, describes them by split (Table 1), and records the data checks behind the feature choices. Only aggregate counts are shown.

- Exploration of the outcome uses the training years only; for the test year (2024) we show feature distributions but **not** outcome rates.
- Race and ethnicity are never features; they appear here only to describe missingness.

In [1]:
%load_ext autoreload
%autoreload 2
import sys; sys.path.append("..")
import pandas as pd, numpy as np
from src import config
from src.features import FEATURE_SETS, NUMERIC, CATEGORICAL, check_leakage, get_X
pd.set_option("display.width", 140); pd.set_option("display.max_rows", 200)
df = pd.read_parquet(config.ANALYTIC_FILE)   # built by: python -m src.build_dataset
main = df[df[config.MAIN_SAMPLE_COL]].copy()
print(f"main sample: {len(main):,} rows;", main[config.SPLIT_COL].value_counts().to_dict())

main sample: 449,474 rows; {'train': 193196, 'test': 135821, 'valid': 120457}


## 1. Feature sets

Each set adds to the previous one. Gender is an audit variable, like race; the supplementary set `CASE_PLUS_GENDER` (team decision, 2026-11-07) adds it to `CASE` only to measure how much it adds beyond case and record. `check_leakage` rejects any column on `config.LEAKAGE_BLOCKLIST` (the decision itself, anything recorded after arraignment, defense-counsel type, prosecutor requests), any `y_*` outcome and the fairness variables.

In [2]:
from src.features import SUPPLEMENTARY_SETS
for name, cols in FEATURE_SETS.items():
    check_leakage(cols)
    print(f"{name} ({len(cols)}): {cols}")
for name, cols in SUPPLEMENTARY_SETS.items():
    print(f"[supplementary] {name} ({len(cols)}): {cols}")
print("\nNumeric:", NUMERIC)
X = get_X(main, "CASE_COURT_JUDGE").join(get_X(main, "CASE_PLUS_GENDER")[["gender"]])   # all features, for description
print("\nModel-ready frame:", X.shape, "| categorical levels:", {c: X[c].nunique() for c in CATEGORICAL})

CASE (18): ['top_severity_at_arraign', 'top_charge_weight_at_arraign', 'top_charge_at_arraign_violent_felony_ind', 'top_arraign_attempt_indicator', 'arraign_charge_category', 'hate_crime_ind', 'arrest_type', 'case_type', 'prior_vfo_cnt', 'prior_nonvfo_cnt', 'prior_misd_cnt', 'pend_vfo', 'pend_nonvfo', 'pend_misd', 'supervision', 'dcjs_missing', 'dat_wo_ws_prior_to_arraign', 'age_at_arrest']
CASE_COURT (19): ['top_severity_at_arraign', 'top_charge_weight_at_arraign', 'top_charge_at_arraign_violent_felony_ind', 'top_arraign_attempt_indicator', 'arraign_charge_category', 'hate_crime_ind', 'arrest_type', 'case_type', 'prior_vfo_cnt', 'prior_nonvfo_cnt', 'prior_misd_cnt', 'pend_vfo', 'pend_nonvfo', 'pend_misd', 'supervision', 'dcjs_missing', 'dat_wo_ws_prior_to_arraign', 'age_at_arrest', 'county_name']
CASE_COURT_JUDGE (20): ['top_severity_at_arraign', 'top_charge_weight_at_arraign', 'top_charge_at_arraign_violent_felony_ind', 'top_arraign_attempt_indicator', 'arraign_charge_category', 'hat


Model-ready frame: (449474, 21) | categorical levels: {'top_severity_at_arraign': 2, 'top_charge_weight_at_arraign': 8, 'top_charge_at_arraign_violent_felony_ind': 2, 'top_arraign_attempt_indicator': 2, 'arraign_charge_category': 21, 'hate_crime_ind': 2, 'arrest_type': 2, 'case_type': 2, 'dat_wo_ws_prior_to_arraign': 2, 'county_name': 5, 'judge_id': 186, 'gender': 3}


## 2. Table 1 — main sample by split

Categorical features: number and share of rows in each level. Numeric features: mean, median and share missing. Levels with fewer than `config.MIN_GROUP_N` rows in any split are suppressed (shown as missing). `y_detain` is computed for training and validation only; the test-year outcome is not computed before the freeze. Saved to `reports/table1.csv`.

In [3]:
splits = ["train", "valid", "test"]
sp = main[config.SPLIT_COL]
rows = []
for c in [c for c in CATEGORICAL if c != "judge_id"]:
    n = pd.crosstab(X[c], sp).reindex(columns=splits, fill_value=0)
    share = n / n.sum()
    for level in n.index:
        small = (n.loc[level] < config.MIN_GROUP_N).any()
        row = {"feature": c, "level": level, "stat": "share"}
        for s_ in splits:
            row[s_] = np.nan if small else round(share.loc[level, s_], 4)
            row[f"n_{s_}"] = np.nan if small else int(n.loc[level, s_])
        rows.append(row)
for c in NUMERIC:
    g = X[c].groupby(sp)
    for stat, vals in [("mean", g.mean()), ("median", g.median()), ("share missing", X[c].isna().groupby(sp).mean())]:
        rows.append({"feature": c, "level": "", "stat": stat, **{s_: round(vals[s_], 4) for s_ in splits}})
jk = main["judge_id"].where(~main["judge_id"].str.startswith("J").fillna(False), "coded")
for level in ["coded", "other", "unseen"]:
    rows.append({"feature": "judge_id", "level": level, "stat": "share", **{s_: round((jk[sp == s_] == level).mean(), 4) for s_ in splits}})
tv = main[sp.isin(["train", "valid"])]                     # outcome: training and validation only
y = tv.groupby(config.SPLIT_COL)[config.TARGET_Q1].mean()
rows.append({"feature": "y_detain", "level": "1", "stat": "share", "train": round(y["train"], 4), "valid": round(y["valid"], 4), "test": np.nan})
rows.append({"feature": "N", "level": "", "stat": "rows", **{s_: int((sp == s_).sum()) for s_ in splits}})
table1 = pd.DataFrame(rows)[["feature", "level", "stat"] + splits + [f"n_{s_}" for s_ in splits]]
table1.to_csv(config.REPORTS / "table1.csv", index=False)
print("Suppressed levels (< MIN_GROUP_N in some split):", table1.loc[(table1["stat"] == "share") & table1["train"].isna() & (table1["feature"] != "y_detain"), ["feature", "level"]].values.tolist())
table1

Suppressed levels (< MIN_GROUP_N in some split): [['arraign_charge_category', 'Conspiracy'], ['gender', 'Unknown']]


,feature,level,stat,train,valid,test,n_train,n_valid,n_test
0,top_severity_at_arraign,Felony,share,0.3055,0.3187,0.3113,59016.0,38385.0,42283.0
1,top_severity_at_arraign,Misdemeanor,share,0.6945,0.6813,0.6887,134180.0,82072.0,93538.0
2,top_charge_weight_at_arraign,AF,share,0.0077,0.0061,0.0056,1483.0,734.0,757.0
3,top_charge_weight_at_arraign,AM,share,0.6197,0.5951,0.6020,119732.0,71685.0,81761.0
4,top_charge_weight_at_arraign,BF,share,0.0624,0.0645,0.0645,12063.0,7765.0,8756.0
5,top_charge_weight_at_arraign,BM,share,0.0166,0.0178,0.0187,3211.0,2141.0,2539.0
6,top_charge_weight_at_arraign,CF,share,0.0753,0.0699,0.0661,14555.0,8419.0,8978.0
7,top_charge_weight_at_arraign,DF,share,0.1058,0.1157,0.1159,20432.0,13935.0,15740.0
8,top_charge_weight_at_arraign,EF,share,0.0543,0.0625,0.0593,10483.0,7532.0,8052.0
9,top_charge_weight_at_arraign,UM,share,0.0582,0.0685,0.0680,11237.0,8246.0,9238.0


## 3. Missing values and anomalies

### 3.1 Prior-record (DCJS) fields
Missing DCJS fields mean the docket did not match a DCJS criminal-history record, **not** that the person has no record. They are kept as `NaN` plus the `dcjs_missing` indicator.

In [4]:
pd.crosstab(main["county_name"], main[config.YEAR_COL], values=main["dcjs_missing"], aggfunc="mean").round(4)

arraign_year,2021,2022,2023,2024
county_name,,,,
Bronx,0.0067,0.0054,0.0085,0.0146
Kings,0.0107,0.0136,0.0205,0.0247
New York,0.0116,0.0056,0.0060,0.0068
Queens,0.0073,0.0066,0.0090,0.0122
Richmond,0.0071,0.0105,0.0167,0.0272


In [5]:
tr = main[main[config.SPLIT_COL] == "train"]
tr.groupby("dcjs_missing")[config.TARGET_Q1].agg(["size", "mean"]).round(4).rename(columns={"mean": "y_detain"})

,size,y_detain
dcjs_missing,,
0,191482,0.1303
1,1714,0.0163


### 3.2 Race recorded as Unknown (fairness variable, not a feature)

In [6]:
pd.crosstab(main["county_name"], main[config.YEAR_COL], values=main["race"].eq("Unknown").astype(int), aggfunc="mean").round(4)

arraign_year,2021,2022,2023,2024
county_name,,,,
Bronx,0.0270,0.0073,0.0151,0.0059
Kings,0.0473,0.0055,0.0117,0.0030
New York,0.0432,0.0072,0.0177,0.0062
Queens,0.1589,0.0069,0.0157,0.0059
Richmond,0.0383,0.0090,0.0145,0.0077


In [7]:
tr.assign(race_unknown=tr["race"].eq("Unknown")).groupby("race_unknown")[config.TARGET_Q1].agg(["size", "mean"]).round(4).rename(columns={"mean": "y_detain"})

,size,y_detain
race_unknown,,
False,186325,0.1313
True,6871,0.0758


### 3.3 Other categorical fields
In the main sample (local-court dockets), `arrest_type` and `top_severity_at_arraign` have no missing values; the missing values seen in the raw files belong to superior-court cases removed at step S4.

In [8]:
cols = ["arrest_type", "top_severity_at_arraign", "top_charge_weight_at_arraign", "gender", "arraign_charge_category"]
pd.Series({c: int(main[c].isna().sum()) for c in cols}, name="missing rows")

arrest_type                     0
top_severity_at_arraign         0
top_charge_weight_at_arraign    0
gender                          0
arraign_charge_category         0
Name: missing rows, dtype: int64

### 3.4 Age at arrest

In [9]:
age = main["age_at_arrest"]
print(age.describe().round(1).to_dict())
pd.Series({"age < 13": int((age < 13).sum()), "13-15": int(age.between(13, 15).sum()),
           "16-17": int(age.between(16, 17).sum()), "> 90": int((age > 90).sum())}, name="rows")

{'count': 449418.0, 'mean': 35.6, 'std': 12.1, 'min': 13.0, '25%': 26.0, '50%': 33.0, '75%': 43.0, 'max': 95.0}


age < 13       0
13-15       1263
16-17       3945
> 90           7
Name: rows, dtype: int64

In [10]:
print("Rows with age_fixed = 1 by year:", main.groupby(config.YEAR_COL)["age_fixed"].sum().to_dict())
print("Still missing after the fix:", int(age.isna().sum()))
print("Case type of age < 16 rows:", main.loc[age < 16, "case_type"].value_counts().to_dict())

Rows with age_fixed = 1 by year: {np.int64(2021): 440, np.int64(2022): 365, np.int64(2023): 496, np.int64(2024): 597}
Still missing after the fix: 56
Case type of age < 16 rows: {'Felony Youth Complaint': 1263}


In the raw files `age_at_arrest` = 0 for about 0.4% of dockets (a placeholder, not an age). `src/sample.py` replaces it with `age_at_crime` when that is positive and otherwise sets it to missing (`age_fixed` = 1 marks these rows). Remaining ages under 16 are mostly felony youth complaints (adolescent offenders), which is expected.

## 4. Consistency across years

### 4.1 Data dictionary
The dictionary and documentation re-downloaded on 2026-11-07 are byte-identical to the 2026-11-06 copies. The current dictionary has the same six release-decision values; there is no separate "nominal bail" category, so nominal bail is identified from the bail amounts in `src/sample.py`.

### 4.2 Categories present in some years but not others
New levels in the validation or test year are handled by `OneHotEncoder(handle_unknown="ignore")`, but we list them so nothing is silent.

In [11]:
out = []
for c in [c for c in CATEGORICAL if c != "judge_id"]:
    present = main.groupby(config.YEAR_COL)[c].apply(lambda s: set(s.dropna().unique()))
    allv = set().union(*present)
    for v in sorted(allv):
        yrs = [y for y in present.index if v in present[y]]
        if len(yrs) < len(present):
            out.append({"feature": c, "level": v, "years_present": yrs, "rows": int((main[c] == v).sum())})
pd.DataFrame(out) if out else "No feature level appears in only some years."


'No feature level appears in only some years.'

### 4.3 Feature means by year (training years only show the outcome)

In [12]:
main.groupby(config.YEAR_COL)[NUMERIC].mean().round(3)

,prior_vfo_cnt,prior_nonvfo_cnt,prior_misd_cnt,pend_vfo,pend_nonvfo,pend_misd,supervision,dcjs_missing,age_at_arrest
arraign_year,,,,,,,,,
2021,0.212,0.442,1.664,0.097,0.162,0.204,0.058,0.009,35.214
2022,0.222,0.449,1.651,0.097,0.162,0.201,0.055,0.009,35.507
2023,0.210,0.410,1.501,0.088,0.158,0.190,0.048,0.012,35.559
2024,0.203,0.395,1.457,0.087,0.159,0.193,0.043,0.016,35.858


## 5. Notes for the team

- `get_X(df, feature_set)` returns the model-ready frame; numeric `NaN`s must be imputed **inside** the pipeline (fitted on training data only).
- For the final refit before the 2024 test, call `get_X(df, feature_set, final=True)` so `judge_id` uses the 2021–2023 coding (`judge_id_final`).
- Gender: audit variable in the main sets; `CASE_PLUS_GENDER` is reported as a supplementary comparison, and the final choice is made after seeing it.
- 2021 race = Unknown is 15.9% in Queens (about 4% elsewhere, about 1% from 2022): fairness results for 2021 should note this.

## 6. A2-05: Proxy variables — preliminary training-only evidence

**Scope:** 2021–2022 main-sample dockets only. The table describes associations with recorded race; race is a fairness grouping variable, never a model input. These are docket-level, unadjusted associations, not estimates of policing effects, bias, or causation. Race Unknown is retained. Prior counts are capped in the source and missing DCJS records are excluded from the prior-count denominator, not treated as zero. Suppress groups and numerator/complement cells below `MIN_GROUP_N`.

- **Prior violent-felony convictions:** potentially reflects accumulated criminal-justice contact as well as prior convictions. Differences across recorded race are descriptive and cannot establish why they arise. See the [National Academies report on racial inequality in crime and justice](https://nap.nationalacademies.org/skim.php?chap=139-186&record_id=26705).
- **Arrest type (DAT versus custody):** may encode charge eligibility, identification, warrants, and policing or administrative processes. It is not a pure measure of police discretion: the [NYPD description](https://www.nyc.gov/site/nypd/services/victim-services/criminal-justice-process.page) describes eligibility and identification. The table does not isolate their effects.
- **Age:** may correlate with accumulated exposure to criminal-justice contact and life stage; the association shown here does not establish a mechanism.
- **Gender:** may correlate with patterns of criminal-justice contact and social treatment; the data record gender categories, not individual gender identity or a causal mechanism. Its inclusion remains a team decision.

This is a draft for the later slide, not a recommendation to remove any feature. Check whether the timing of prior-record fields is documented before presenting them as information available at arraignment.


In [13]:
# Read only training rows and selected columns; do not access the test split.
proxy_cols = [config.SPLIT_COL, config.MAIN_SAMPLE_COL, "race", "prior_vfo_cnt",
              "arrest_type", "age_at_arrest", "gender"]
proxy = pd.read_parquet(config.ANALYTIC_FILE, columns=proxy_cols,
                        filters=[(config.SPLIT_COL, "==", "train"),
                                 (config.MAIN_SAMPLE_COL, "==", True)])

def safe_share(mask, eligible):
    n = int(eligible.sum())
    k = int((mask & eligible).sum())
    if n < config.MIN_GROUP_N or k < config.MIN_GROUP_N or n - k < config.MIN_GROUP_N:
        return np.nan
    return round(k / n, 3)

proxy_rows = []
for race, group in proxy.groupby("race", dropna=False):
    if len(group) < config.MIN_GROUP_N:
        continue
    prior_observed = group["prior_vfo_cnt"].notna()
    all_rows = pd.Series(True, index=group.index)
    age_observed = group["age_at_arrest"].notna()
    proxy_rows.append({
        "recorded_race": race, "dockets": len(group),
        "prior_vfo_gt_0_among_observed": safe_share(group["prior_vfo_cnt"].gt(0), prior_observed),
        "prior_vfo_missing": safe_share(~prior_observed, all_rows),
        "dat_share": safe_share(group["arrest_type"].eq("DAT"), all_rows),
        "mean_age_observed": round(group.loc[age_observed, "age_at_arrest"].mean(), 1)
            if age_observed.sum() >= config.MIN_GROUP_N else np.nan,
        "female_share": safe_share(group["gender"].eq("Female"), all_rows),
    })
pd.DataFrame(proxy_rows).set_index("recorded_race")


,dockets,prior_vfo_gt_0_among_observed,prior_vfo_missing,dat_share,mean_age_observed,female_share
recorded_race,,,,,,
American Indian/Alaskan Native,345,NaN,NaN,NaN,37.3,NaN
Asian/Pacific Islander,6715,0.034,NaN,0.118,37.4,0.145
Black,111070,0.219,0.008,0.136,34.7,0.185
Unknown,6871,0.050,0.015,0.196,36.7,0.162
White,68183,0.122,0.010,0.166,36.1,0.186


## 7. Slide copy for the team deck

### Features and leakage rules

- **CASE:** charges, prior convictions, pending cases, supervision, prior DAT failure to appear, and age. Gender is reserved for auditing; `CASE_PLUS_GENDER` is a supplementary comparison.
- **CASE + COURT:** add county. **CASE + COURT + JUDGE:** add anonymized judge ID, coded on the training years for validation.
- Exclude the decision, bail amounts, later outcomes, the most recent attorney type, race, ethnicity, and gender in the main sets. These variables are reserved for fairness analysis, with gender allowed in `CASE_PLUS_GENDER`. Confirm prior-record timing against the source documentation.

### Potential proxy variables

In the 2021–2022 NYC main sample, 21.9% of dockets recorded as Black and 12.2% recorded as White have a prior violent-felony conviction among those with observed prior counts. DAT shares are 13.6% and 16.6%; mean ages are 34.7 and 36.1; recorded female shares are 18.5% and 18.6%, respectively. These comparisons are unadjusted, describe dockets rather than unique people, and do not establish why differences exist. Prior counts are capped; missing prior records are excluded from the prior-count denominator.

Prior convictions may reflect accumulated justice-system contact as well as conviction history. DAT status also reflects eligibility and administrative procedures. Age and recorded gender may correlate with life stage and social treatment. The data do not identify these mechanisms. Sources: UCS Pretrial Release Data (2021–22); [NYPD Criminal Justice Process](https://www.nyc.gov/site/nypd/services/victim-services/criminal-justice-process.page); [National Academies, Reducing Racial Inequality in Crime and Justice](https://nap.nationalacademies.org/skim.php?chap=139-186&record_id=26705).
